# Exploration du jeu de développement

Ce notebook ne lit **que** `data/processed/developpement.parquet` (steps 1 à 355).
Le jeu de test n'est jamais ouvert ici (décisions D8 et D12).

In [2]:
import os
print(os.getcwd())

/home/onel/projets/fraude-mobile-money/notebooks


In [3]:
import duckdb
import polars as pl

pl.Config.set_tbl_rows(40)

con = duckdb.connect()
con.execute("CREATE VIEW dev AS SELECT * FROM read_parquet('../data/processed/developpement.parquet')")
con.sql("SELECT COUNT(*) AS transactions, MIN(step) AS step_min, MAX(step) AS step_max FROM dev").pl()

transactions,step_min,step_max
i64,i32,i32
5113884,1,355


je pense pas que 100 % des fraudes sont dans TRANSFER et CASH_OUT,

In [4]:
con.sql("""
    SELECT
        type,
        COUNT(*) AS transactions,
        SUM(isFraud) AS fraudes,
        ROUND(100.0 * SUM(isFraud) / COUNT(*), 3) AS taux_fraude_pct,
        ROUND(100.0 * SUM(isFraud) / SUM(SUM(isFraud)) OVER (), 2) AS part_des_fraudes_pct
    FROM dev
    GROUP BY type
    ORDER BY fraudes DESC
""").pl()

type,transactions,fraudes,taux_fraude_pct,part_des_fraudes_pct
str,i64,"decimal[38,0]",f64,f64
"""CASH_OUT""",1813999,1991,0.11,50.24
"""TRANSFER""",424470,1972,0.465,49.76
"""PAYMENT""",1720408,0,0.0,0.0
"""DEBIT""",32533,0,0.0,0.0
"""CASH_IN""",1122474,0,0.0,0.0


In [5]:
con.sql("""
    SELECT
        isFraud,
        COUNT(*) AS transactions,
        ROUND(100.0 * AVG(CASE WHEN ABS(amount - oldbalanceOrg) < 0.01 THEN 1 ELSE 0 END), 2) AS montant_egal_solde_pct,
        ROUND(100.0 * AVG(CASE WHEN newbalanceOrig = 0 THEN 1 ELSE 0 END), 2) AS solde_final_zero_pct,
        ROUND(100.0 * AVG(CASE WHEN oldbalanceOrg = 0 THEN 1 ELSE 0 END), 2) AS solde_initial_zero_pct
    FROM dev
    WHERE type IN ('TRANSFER', 'CASH_OUT')
    GROUP BY isFraud
    ORDER BY isFraud
""").pl()

isFraud,transactions,montant_egal_solde_pct,solde_final_zero_pct,solde_initial_zero_pct
i8,i64,f64,f64,f64
0,2234506,0.0,90.14,47.73
1,3963,98.13,98.79,0.71


In [6]:
con.sql("""
    SELECT COUNT(*) AS paires_trouvees
    FROM dev t
    JOIN dev c
      ON t.step = c.step
     AND ABS(t.amount - c.amount) < 0.01
    WHERE t.type = 'TRANSFER' AND t.isFraud = 1
      AND c.type = 'CASH_OUT' AND c.isFraud = 1
""").pl()

paires_trouvees
i64
1993


In [7]:
con.sql("""
    SELECT
        type,
        isFraud,
        COUNT(*) AS transactions,
        ROUND(MEDIAN(amount), 0) AS montant_median,
        ROUND(MAX(amount), 0) AS montant_max
    FROM dev
    WHERE type IN ('TRANSFER', 'CASH_OUT')
    GROUP BY type, isFraud
    ORDER BY type, isFraud
""").pl()

type,isFraud,transactions,montant_median,montant_max
str,i8,i64,f64,f64
"""CASH_OUT""",0,1812008,147988.0,2.847567e6
"""CASH_OUT""",1,1991,405062.0,1e7
"""TRANSFER""",0,422498,489100.0,9.2445517e7
"""TRANSFER""",1,1972,417954.0,1e7
